# Fabric 01 · The cheap baseline — RAG over the rules (no training)

**De-risk before you fine-tune.** This notebook grounds the **base** model on the abstraction rules via retrieval — rules live as *editable text*, not weights, so a policy change is a one-line edit. It sets the **bar**: whatever RAG *can't* crack is exactly what justifies fine-tuning.

> Cheapest to stand up, easiest to govern, lowest ceiling on the hard Rule 1 conflict cases.

In [ ]:
# === Dual-mode setup: Microsoft Fabric (Spark + Lakehouse) OR local (pandas + repo files) ===
import os, json
from pathlib import Path

try:
    import notebookutils            # exists ONLY inside Microsoft Fabric
    IN_FABRIC = True
except Exception:
    IN_FABRIC = False

def _find_data_dir():
    here = Path.cwd()
    for c in [here, *here.parents]:
        d = c / 'fine-tuning' / 'data'
        if d.exists():
            return d
    return Path('fine-tuning/data')
DATA_DIR = None if IN_FABRIC else _find_data_dir()

AZURE_OPENAI_ENDPOINT = os.environ.get('AZURE_OPENAI_ENDPOINT', 'https://<your-foundry>.cognitiveservices.azure.com/')
API_VERSION           = os.environ.get('AZURE_OPENAI_API_VERSION', '2025-04-01-preview')
BASE_DEPLOYMENT       = os.environ.get('BASE_DEPLOYMENT', 'gpt-4o-mini')
TUNED_DEPLOYMENT      = os.environ.get('TUNED_DEPLOYMENT', 'acme-rtor-deployment')

# Entra token for Azure OpenAI: Fabric token broker in-cloud, DefaultAzureCredential locally.
if IN_FABRIC:
    def _token():
        return notebookutils.credentials.getToken('https://cognitiveservices.azure.com')
else:
    from azure.identity import DefaultAzureCredential
    _cred = DefaultAzureCredential()
    def _token():
        return _cred.get_token('https://cognitiveservices.azure.com/.default').token

from openai import AzureOpenAI
client = AzureOpenAI(
    azure_endpoint          = AZURE_OPENAI_ENDPOINT,
    azure_ad_token_provider = _token,
    api_version             = API_VERSION,
)
print('mode    :', 'FABRIC' if IN_FABRIC else 'LOCAL')
print('endpoint:', AZURE_OPENAI_ENDPOINT)
print('models  : base=' + BASE_DEPLOYMENT + '  tuned=' + TUNED_DEPLOYMENT)


In [ ]:
# === The RTOR abstraction prompt + defensive parser (identical to the Foundry labs) ===
import json

RULES_BLOCK = '''
### SPECIFIC ABSTRACTION RULES

Rule 1 - Conflict-resolution order (apply in this EXACT priority; the FIRST match decides):
  1. Planned / staged overrides everything (planned/staged/anticipated/scheduled at index) -> false, even within 30 days.
  2. Unplanned + related complication (bleeding, hematoma, SSI, dehiscence, anastomotic leak, abscess, graft/flap failure) within 30 days -> true.
  3. Unrelated anatomy or new diagnosis -> false, regardless of timing.
  4. Outside the 30-day window -> false.

Rule 2 - Operating-room requirement. Bedside / ICU / IR / endoscopy-suite / clinic procedures do NOT count -> false.

Rule 3 - Evidence requirement. Quote the single most decisive sentence verbatim, then state which rule it triggers.
'''

SYSTEM_PROMPT = (
    'You are a surgical-quality abstraction assistant for Acme Health. Determine whether the '
    'current operative episode is an unplanned Return to the Operating Room (RTOR) for the index '
    'surgery, applying the rules below.\n'
    + RULES_BLOCK +
    '\n### TASK EXECUTION\n'
    '- Read the provided text thoroughly.\n'
    '- Resolve conflicting data using the exact order in Rule 1.\n'
    '- Output ONLY a valid JSON object with exactly two keys: "is_return_to_or" (boolean) and '
    '"evidence" (string citing the exact text used and how it applies to the rules).\n'
    '- No conversational filler. No markdown json fence.'
)

TEMPLATE = '''Patient Timeline:
{patient_timeline_json}

Progress Note Details:
{progress_note_json}

Index Surgery Procedure Description:
{index_surgery_procedure_desc}

Index Surgery Operative Note:
{index_surgery_op_note}

Current Surgery Procedure Description:
{current_surgery_procedure_desc}

Current Surgery Operative Note:
{current_surgery_op_note}

Task: Determine if the current operating note/surgery represents a return to the operating room based strictly on the abstraction rules provided above. Output ONLY the raw JSON object.'''

def build_user_prompt(case):
    return TEMPLATE.format(
        patient_timeline_json          = json.dumps(case.get('patient_timeline', []), indent=2),
        progress_note_json             = json.dumps(case.get('progress_note', {}), indent=2),
        index_surgery_procedure_desc   = case.get('index_surgery_procedure_desc', ''),
        index_surgery_op_note          = case.get('index_surgery_op_note', ''),
        current_surgery_procedure_desc = case.get('current_surgery_procedure_desc', ''),
        current_surgery_op_note        = case.get('current_surgery_op_note', ''),
    )

def safe_parse(val):
    '''Extract JSON from the model response, stripping stray markdown fences.'''
    try:
        clean = str(val).strip()
        if clean.startswith('```'):
            clean = clean.strip('`')
            if clean.startswith('json'):
                clean = clean[4:]
        return json.loads(clean.strip())
    except Exception as e:
        return {'is_return_to_or': None, 'evidence': f'Parse Error: {e} | Raw: {val}'}

print('prompt + parser ready')


In [ ]:
# === Load the eval cases (full dicts incl. gold_*). Fabric -> Lakehouse table; local -> jsonl ===
def load_eval():
    if IN_FABRIC:
        rows = spark.read.table('surgical_episodes').toPandas().to_dict('records')
        return [json.loads(r['case_json']) for r in rows]
    p = DATA_DIR / 'rtor_eval.jsonl'
    assert p.exists(), f'missing {p} -- run the Foundry clinical Lab 00 first'
    return [json.loads(l) for l in p.read_text(encoding='utf-8').splitlines() if l.strip()]

EVAL = load_eval()
print('eval cases:', len(EVAL))


In [ ]:
# === Persist / load predictions so the eval notebook can compare every approach ===
def save_preds(name, preds):
    if IN_FABRIC:
        import pandas as pd
        (spark.createDataFrame(pd.DataFrame(preds))
            .write.format('delta').mode('overwrite').saveAsTable(f'preds_{name}'))
    else:
        (DATA_DIR / f'preds_{name}.json').write_text(json.dumps(preds), encoding='utf-8')
    print(f'saved preds -> {name} ({len(preds)})')

def load_preds(name):
    if IN_FABRIC:
        return spark.read.table(f'preds_{name}').toPandas().to_dict('records')
    return json.loads((DATA_DIR / f'preds_{name}.json').read_text(encoding='utf-8'))


In [ ]:
# === Shared scoring: classification metrics + LLM-as-judge evidence groundedness ===
def score(preds):
    tp = sum(1 for p in preds if p['gold'] and p['pred'] is True)
    tn = sum(1 for p in preds if (not p['gold']) and p['pred'] is False)
    fp = sum(1 for p in preds if (not p['gold']) and p['pred'] is True)
    fn = sum(1 for p in preds if p['gold'] and p['pred'] is False)
    un = sum(1 for p in preds if p['pred'] is None)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    acc  = (tp + tn) / len(preds) if preds else 0.0
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,
            'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn, 'unparsed': un}

def judge_groundedness(preds):
    scored = [p for p in preds if p.get('pred') is not None and p.get('pred_evidence')]
    if not scored:
        return 0.0
    total = 0
    for p in scored:
        sysmsg = ('You grade whether an abstraction citation is well-grounded. Given GOLD evidence and '
                  'MODEL evidence for a surgical Return-to-OR decision, reply JSON {"score": 0 or 1}. '
                  'score=1 means the model quoted a relevant source sentence and named a plausible rule.')
        r = client.chat.completions.create(model=BASE_DEPLOYMENT, temperature=0.0, max_tokens=80,
            response_format={'type': 'json_object'},
            messages=[{'role': 'system', 'content': sysmsg},
                      {'role': 'user', 'content': json.dumps({'gold': p.get('gold_evidence'), 'model': p.get('pred_evidence')})}])
        total += int(safe_parse(r.choices[0].message.content).get('score', 0) or 0)
    return total / len(scored)

def print_board(name, m):
    print(f'[{name}]  acc={m["accuracy"]:.0%}  prec={m["precision"]:.0%}  rec={m["recall"]:.0%}  '
          f'f1={m["f1"]:.0%}  (TP={m["tp"]} TN={m["tn"]} FP={m["fp"]} FN={m["fn"]} unparsed={m["unparsed"]})')


---
## Step 1 — Index the rules as retrievable chunks

The rules KB is chunked by section. A real deployment swaps the keyword scorer for an embedding index; the *pattern* — retrieve the relevant rules, inject them — is identical.

In [ ]:
RULE_CHUNKS = [{"rule_id": "Return to the Operating Room (RTOR) — Clinical A", "text": "# Return to the Operating Room (RTOR) — Clinical Abstraction Rules\n\n> **Example data for a clinical document-abstraction scenario.** Every patient\n> timeline, operative note, and provider NPI below is **fully synthetic** and\n> contains **no real PHI**. Replace the rules and notes with your own registry\n> definition when you customize the accelerator.\n\nThis knowledge base defines how a surgical-quality abstractor decides whether a\ngiven operative episode counts as an **unplanned Return to the Operating Room\n(RTOR)** for the index surgical procedure. It is consumed by the **Acme Health\nSurgical Quality abstraction assistant**, which reads the structured inputs for\none case and emits `{ \"is_return_to_or\": <bool>, \"evidence\": \"<exact text>\" }`.\n\nThe model must apply the rules **strictly** and cite the **exact sentence** from\nthe source documents that justified its determination.\n\n---"}, {"rule_id": "Inputs available per case", "text": "## Inputs available per case\n\nFor each case the abstractor receives:\n\n- **Patient timeline** — dated surgical events for this patient (the index\n  surgery plus any subsequent procedures), with the day offset from the index\n  surgery.\n- **Progress note** — the clinical narrative around the current encounter.\n- **Index surgery procedure description** — the short CPT-style description of\n  the original (index) operation.\n- **Index surgery operative note** — the narrative of the index operation,\n  including any documented plan to return.\n- **Current surgery procedure description** — the short description of the\n  operation under review.\n- **Current surgery operative note** — the narrative of the operation under\n  review, including the stated indication.\n\n---"}, {"rule_id": "Definition", "text": "## Definition\n\nA **Return to the Operating Room (RTOR)** is an **unplanned** return to an\noperating room for a surgical procedure that is **related to the index surgery**\nand occurs within **30 days** of the index surgery.\n\nA return is **not** an RTOR if it was **planned or staged** at the time of the\nindex surgery, if it is **unrelated** to the index surgery (different anatomy,\nnew diagnosis), or if it occurs **more than 30 days** after the index surgery.\n\n---"}, {"rule_id": "Specific Abstraction Rules", "text": "## Specific Abstraction Rules"}, {"rule_id": "Rule 1 — Conflict-resolution order (apply in thi", "text": "### Rule 1 — Conflict-resolution order (apply in this exact priority)\n\nWhen more than one rule could apply, resolve the conflict by applying the rules\nin the following order. The **first** rule that matches decides the case.\n\n1. **Planned / staged overrides everything.** If the index operative note OR the\n   current operative note documents that the second procedure was **planned,\n   staged, anticipated, or scheduled** at the time of the index surgery\n   (e.g. \"planned second-look\", \"staged washout\", \"return to OR scheduled for\n   delayed closure\"), then **`is_return_to_or = false`** — even if it occurs\n   within 30 days. A planned return is part of the original surgical plan.\n\n2. **Unplanned + related + within 30 days = RTOR.** If the current surgery is\n   **unplanned** and addresses a **complication of the index surgery**\n   (post-operative bleeding, hematoma, surgical-site infection, wound\n   dehiscence, anastomotic leak, abscess, graft/flap failure, retained foreign\n   body) **and** occurs within 30 days of the index surgery, then\n   **`is_return_to_or = true`**.\n\n3. **Unrelated anatomy or new diagnosis = not RTOR.** If the current surgery is\n   for a **different anatomic site or a new, unrelated diagnosis** (e.g. index\n   was a knee arthroplasty, current is an appendectomy), then\n   **`is_return_to_or = false`**, regardless of timing.\n\n4. **Outside the 30-day window = not RTOR.** If the current surgery occurs\n   **more than 30 days** after the index surgery and is not otherwise captured\n   above, then **`is_return_to_or = false`**."}, {"rule_id": "Rule 2 — \"Operating room\" requirement", "text": "### Rule 2 — \"Operating room\" requirement\n\nThe return must be to an **operating room**. Bedside procedures, interventional\nradiology suites, endoscopy suites, and clinic-based procedures do **not** count\nas a return to the OR. If the current note documents the procedure was performed\n**at the bedside / in the ICU / in IR / in the endoscopy suite**, then\n**`is_return_to_or = false`**."}, {"rule_id": "Rule 3 — Evidence requirement", "text": "### Rule 3 — Evidence requirement\n\nThe `evidence` field must quote the **single most decisive sentence** from the\nsource documents (operative notes, progress note, or timeline) that supports the\ndetermination. Do not paraphrase the clinical facts; cite the source text and\nthen state which rule it triggers.\n\n---"}, {"rule_id": "Worked examples", "text": "## Worked examples\n\n- **Unplanned reoperation for bleeding (day 2)** → Rule 2 → `true`. Evidence:\n  \"Taken back to the OR emergently for evacuation of an expanding hematoma.\"\n- **Staged abdominal washout documented at index** → Rule 1 → `false`. Evidence:\n  \"Abdomen left open; planned return to OR in 48 hours for second-look washout\n  and delayed fascial closure.\"\n- **Appendectomy 9 days after a total knee arthroplasty** → Rule 1.3 → `false`.\n  Evidence: \"Indication: acute appendicitis,\" unrelated to the index knee.\n- **Reoperation for surgical-site infection on day 41** → Rule 1.4 → `false`.\n  Outside the 30-day window.\n- **Bedside re-exploration in the ICU for fascial dehiscence** → Rule 2 →\n  `false`. Not performed in an operating room."}]

import re
def retrieve_rules(case, k=3):
    """Pick the k most relevant rule chunks for this case (keyword overlap).
    Swap in embeddings (text-embedding-3-small) for a large rule base."""
    blob = ' '.join([
        case.get('index_surgery_op_note', ''),
        case.get('current_surgery_op_note', ''),
        str(case.get('progress_note', '')),
        case.get('index_surgery_procedure_desc', ''),
        case.get('current_surgery_procedure_desc', ''),
    ]).lower()
    def s(ch):
        toks = set(re.findall(r'[a-z]{5,}', ch['text'].lower()))
        return sum(1 for t in toks if t in blob)
    return sorted(RULE_CHUNKS, key=s, reverse=True)[:k]

print('rule chunks indexed:', len(RULE_CHUNKS))


---
## Step 2 — Classify each case with retrieved-rule grounding

In [ ]:
def rag_classify(case):
    chunks = retrieve_rules(case, k=3)
    grounding = '\n\n'.join(c['text'] for c in chunks)
    sysmsg = SYSTEM_PROMPT + '\n\n### RETRIEVED RULES (most relevant to this case)\n' + grounding
    r = client.chat.completions.create(model=BASE_DEPLOYMENT, temperature=0.0, max_tokens=300,
        response_format={'type': 'json_object'},
        messages=[{'role': 'system', 'content': sysmsg},
                  {'role': 'user',   'content': build_user_prompt(case)}])
    return safe_parse(r.choices[0].message.content)

rag_preds = []
for c in EVAL:
    p = rag_classify(c)
    rag_preds.append({'case_id': c['case_id'], 'gold': bool(c['gold_is_return_to_or']),
                      'pred': p.get('is_return_to_or'), 'pred_evidence': p.get('evidence'),
                      'gold_evidence': c.get('gold_evidence')})
save_preds('rag', rag_preds)
print('scored', len(rag_preds), 'cases via RAG baseline')


---
## Step 3 — Score the baseline

In [ ]:
m = score(rag_preds)
print_board('RAG baseline (base model + retrieved rules)', m)
print('evidence groundedness:', f"{judge_groundedness(rag_preds):.0%}")


---
## Takeaways

- RAG gives a **governed, no-training** baseline you can ship today and edit as text.
- Its misses (especially the staged-within-30-days conflict) are the **business case** for fine-tuning — measured, not assumed.
- Next: **Fabric 02** scores the fine-tuned model against this same set, at scale.